# Fair Code — Audit 04: Insurance Denial Bias

> *Young patients and women were flagged for high-cost claims at higher rates than equally healthy older men. BMI, smoking, and diabetic status sound clinical. They're not neutral.*

**Dataset:** Insurance Claim Analysis: Demographic & Health — `insurance.csv`  
**Protected attributes:** Age, Gender  
**Proxy variables:** BMI, Smoker, Diabetic  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

Pipeline:
1. Load and explore the dataset
2. Identify proxy variables
3. Train the biased model (protected attributes + proxies included)
4. Measure the fairness gap (age and gender)
5. Train the fair model (all removed)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import chi2_contingency

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'   # Fair Code gold
DANGER = '#9b2335'   # red — bias
SAFE   = '#4a7c6f'   # teal — mitigated
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('Insurance Denial/insurance.csv')
print(f'Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

In [ ]:
# Target: flag claims above the median as "high-cost" (1 = flagged, 0 = approved)
median_charge = df['claim'].median()
df['high_cost'] = (df['claim'] > median_charge).astype(int)

# Protected attributes
df['age_group'] = df['age'].apply(lambda x: 'Young (<35)' if x < 35 else 'Older (35+)')

print(f'Median claim threshold: ${median_charge:,.2f}')
print(f'\nOverall high-cost flag rate: {df["high_cost"].mean():.1%}')

print('\nHigh-cost flag rate by age group:')
print((df.groupby('age_group')['high_cost'].mean() * 100).round(2).to_string())

print('\nHigh-cost flag rate by gender:')
print((df.groupby('gender')['high_cost'].mean() * 100).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Flag rate by age group
age_rates = df.groupby('age_group')['high_cost'].mean() * 100
ax_colors = [DANGER if g == 'Young (<35)' else MUTED for g in age_rates.index]
axes[0].bar(age_rates.index, age_rates.values, color=ax_colors, width=0.4)
for i, (g, v) in enumerate(age_rates.items()):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[0].set_title('High-cost flag rate by age group (raw data)', color=MUTED, fontsize=10)
axes[0].set_ylabel('High-cost flag rate (%)')
axes[0].set_ylim(0, 75)
axes[0].grid(axis='y', alpha=0.3)

# Flag rate by gender
gender_rates = df.groupby('gender')['high_cost'].mean() * 100
g_colors = [MUTED, DANGER]
axes[1].bar(gender_rates.index, gender_rates.values, color=g_colors, width=0.4)
for i, (g, v) in enumerate(gender_rates.items()):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[1].set_title('High-cost flag rate by gender (raw data)', color=MUTED, fontsize=10)
axes[1].set_ylabel('High-cost flag rate (%)')
axes[1].set_ylim(0, 75)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Identify the Proxy Variables

BMI, smoking status, and diabetic status sound like objective clinical measurements.  
But each one carries encoded signal for race, income, and class — protected characteristics  
that are illegal to use in insurance decisions under the ACA.

| Proxy | Why it encodes protected signal |
|---|---|
| **BMI** | Population BMI distributions differ by race/ethnicity; flagging high BMI penalises race |
| **Smoker** | Smoking correlates with income and education → encodes class and race |
| **Diabetic** | Black and Hispanic Americans diagnosed at 60–100% higher rates; encodes racial disparities in healthcare access |

In [ ]:
print('BMI distribution by age group:')
print(df.groupby('age_group')['bmi'].agg(['mean', 'std']).round(2).to_string())

print('\nSmoker rates by age group:')
smoker_age = pd.crosstab(df['smoker'], df['age_group'], normalize='columns').round(3)
print(smoker_age.to_string())

print('\nDiabetic rates by age group:')
diabetic_age = pd.crosstab(df['diabetic'], df['age_group'], normalize='columns').round(3)
print(diabetic_age.to_string())

In [ ]:
def check_proxy(df, feature, protected_col):
    """Chi-squared test of independence. p < 0.05 = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'p_value': round(p, 6), 'is_proxy': p < 0.05}

print('Chi-squared proxy tests (against age_group):')
for feat in ['smoker', 'diabetic']:
    res = check_proxy(df.dropna(subset=[feat, 'age_group']), feat, 'age_group')
    symbol = '✓ PROXY' if res['is_proxy'] else '✗ not significant'
    print(f"  {feat:<12} p={res['p_value']:.6f}  →  {symbol}")

## 4. Train the Biased Model

Includes `age`, `gender` (protected attributes) and `bmi`, `smoker`, `diabetic` (proxies).

In [ ]:
y = df['high_cost']

X_biased = pd.get_dummies(df[[
    'age',          # protected attribute
    'gender',       # protected attribute
    'bmi',          # proxy: population BMI distributions differ by race
    'bloodpressure',
    'diabetic',     # proxy: diagnosis rates differ significantly by race
    'children',
    'smoker',       # proxy: correlated with income → race/class
    'region',
]])

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
biased_preds = biased_model.predict(X_test)
biased_accuracy = accuracy_score(y_test, biased_preds)

df_test_b = X_test.copy()
df_test_b['age_group']  = df.loc[X_test.index, 'age_group'].values
df_test_b['gender']     = df.loc[X_test.index, 'gender'].values
df_test_b['prediction'] = biased_preds

age_b    = df_test_b.groupby('age_group')['prediction'].mean() * 100
gender_b = df_test_b.groupby('gender')['prediction'].mean() * 100

age_gap_b    = abs(age_b.iloc[0] - age_b.iloc[1])
gender_gap_b = abs(gender_b['male'] - gender_b['female'])

print('--- BIASED MODEL RESULTS ---')
print(f'\nModel Accuracy: {biased_accuracy:.2%}')
print(f'\nHigh-cost flag rate by age group:')
for g, r in age_b.items(): print(f'  {g:<20} {r:.2f}%')
print(f'  Fairness Gap (Age):    {age_gap_b:.2f}%')
print(f'\nHigh-cost flag rate by gender:')
for g, r in gender_b.items(): print(f'  {g:<20} {r:.2f}%')
print(f'  Fairness Gap (Gender): {gender_gap_b:.2f}%')

## 5. Train the Fair Model

Drops `age`, `gender`, `bmi`, `smoker`, `diabetic`.  
Retains only features that reflect documented policy context — not who the person is.

In [ ]:
X_fair = pd.get_dummies(df[[
    'bloodpressure',  # objective clinical measurement
    'children',       # number of dependants — policy-level fact
    'region',         # geographic region — policy-level factor
    # age      removed ✓  (protected attribute)
    # gender   removed ✓  (protected attribute)
    # bmi      removed ✓  (proxy: encodes race via population BMI distributions)
    # smoker   removed ✓  (proxy: encodes income/class → race)
    # diabetic removed ✓  (proxy: diagnosis rates differ 60–100% by race)
]])

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)
fair_preds = fair_model.predict(X_test_f)
fair_accuracy = accuracy_score(y_test_f, fair_preds)

df_test_f = X_test_f.copy()
df_test_f['age_group']  = df.loc[X_test_f.index, 'age_group'].values
df_test_f['gender']     = df.loc[X_test_f.index, 'gender'].values
df_test_f['prediction'] = fair_preds

age_f    = df_test_f.groupby('age_group')['prediction'].mean() * 100
gender_f = df_test_f.groupby('gender')['prediction'].mean() * 100

age_gap_f    = abs(age_f.iloc[0] - age_f.iloc[1])
gender_gap_f = abs(gender_f['male'] - gender_f['female'])

print('--- MITIGATED (UNBIASED) RESULTS ---')
print(f'\nModel Accuracy: {fair_accuracy:.2%}')
print(f'\nHigh-cost flag rate by age group:')
for g, r in age_f.items(): print(f'  {g:<20} {r:.2f}%')
print(f'  New Fairness Gap (Age):    {age_gap_f:.2f}%')
print(f'\nHigh-cost flag rate by gender:')
for g, r in gender_f.items(): print(f'  {g:<20} {r:.2f}%')
print(f'  New Fairness Gap (Gender): {gender_gap_f:.2f}%')

## 6. Compare Results

In [ ]:
age_reduction    = (age_gap_b - age_gap_f) / age_gap_b * 100
gender_reduction = (gender_gap_b - gender_gap_f) / gender_gap_b * 100

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Insurance Denial — Biased vs Mitigated Model', color=ACCENT, fontsize=13)

# Age — biased
age_groups = list(age_b.index)
vals_ab = [age_b.loc[g] for g in age_groups]
colors_ab = [DANGER if 'Young' in g else MUTED for g in age_groups]
bars = axes[0,0].bar(age_groups, vals_ab, color=colors_ab, width=0.4)
for bar, val in zip(bars, vals_ab):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                   f'{val:.1f}%', ha='center', color=ACCENT, fontsize=10)
axes[0,0].set_ylim(0, 85)
axes[0,0].set_title(f'Age — Biased model (Gap: {age_gap_b:.1f}%)', color=MUTED, fontsize=9)
axes[0,0].set_ylabel('High-cost flag rate (%)')
axes[0,0].grid(axis='y', alpha=0.3)

# Age — fair
vals_af = [age_f.loc[g] for g in age_groups]
bars = axes[0,1].bar(age_groups, vals_af, color=[SAFE, SAFE], width=0.4)
for bar, val in zip(bars, vals_af):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                   f'{val:.1f}%', ha='center', color=ACCENT, fontsize=10)
axes[0,1].set_ylim(0, 85)
axes[0,1].set_title(f'Age — Mitigated model (Gap: {age_gap_f:.1f}%)', color=MUTED, fontsize=9)
axes[0,1].set_ylabel('High-cost flag rate (%)')
axes[0,1].grid(axis='y', alpha=0.3)

# Gender — biased
gender_groups = ['female', 'male']
vals_gb = [gender_b.loc[g] for g in gender_groups]
colors_gb = [DANGER, MUTED]
bars = axes[1,0].bar(gender_groups, vals_gb, color=colors_gb, width=0.4)
for bar, val in zip(bars, vals_gb):
    axes[1,0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                   f'{val:.1f}%', ha='center', color=ACCENT, fontsize=10)
axes[1,0].set_ylim(0, 85)
axes[1,0].set_title(f'Gender — Biased model (Gap: {gender_gap_b:.1f}%)', color=MUTED, fontsize=9)
axes[1,0].set_ylabel('High-cost flag rate (%)')
axes[1,0].grid(axis='y', alpha=0.3)

# Gender — fair
vals_gf = [gender_f.loc[g] for g in gender_groups]
bars = axes[1,1].bar(gender_groups, vals_gf, color=[SAFE, SAFE], width=0.4)
for bar, val in zip(bars, vals_gf):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                   f'{val:.1f}%', ha='center', color=ACCENT, fontsize=10)
axes[1,1].set_ylim(0, 85)
axes[1,1].set_title(f'Gender — Mitigated model (Gap: {gender_gap_f:.1f}%)', color=MUTED, fontsize=9)
axes[1,1].set_ylabel('High-cost flag rate (%)')
axes[1,1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Age gap before:         {age_gap_b:.2f}%')
print(f'Age gap after:          {age_gap_f:.2f}%')
print(f'Age gap reduction:      {age_reduction:.1f}%')
print()
print(f'Gender gap before:      {gender_gap_b:.2f}%')
print(f'Gender gap after:       {gender_gap_f:.2f}%')
print(f'Gender gap reduction:   {gender_reduction:.1f}%')

## Key Insight

Insurance AI models don't need to name race, age, or gender to discriminate by them. BMI, smoking status, and diabetic status are the `CustodyStatus` of health insurance — clinical-sounding features that carry protected-class signal because of structural inequalities baked into American healthcare:

- **BMI** is not an independent health signal. It is partially a function of race, ethnicity, and socioeconomic status. Flagging high BMI disproportionately penalises Black and Hispanic patients regardless of whether "race" appears in the feature list.
- **Smoker** status is inversely correlated with income and education, which themselves correlate with race and class. "Smoker" smuggles socioeconomic signal back into the model.
- **Diabetic** status encodes racial disparities in healthcare access and diagnosis rates — not individual risk.

**The fix:** Drop the protected attributes **and** their clinical proxies. Retain only features that reflect policy context (region, number of dependants, blood pressure) rather than encoded group membership.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*